# Ingestion coverage — code-review-graph on `crm-prototype-client`

CRM frontend: React 19 + TanStack Query 5 + TanStack Table 8 + React Router 7 (Vite, `.jsx` / `.js`, JavaScript only).
Fills the team's `Ingestion_Coverage.xlsx` for this project. **Functions expected** come from the TypeScript compiler 5.8.3
(`ref_javascript.py` → `js-ref/functions.mjs`).

Watch for: `queryFn` / `mutationFn` / `onSuccess` in `useQuery` / `useMutation`, TanStack Table column `cell:` functions
(object-property functions, pattern 7), and `forwardRef(...)` / `useCallback(...)` (wrapped, pattern 8).

Every run is a new iteration in `results/crm-prototype-client/runs/iter-NNN_<date>_<time>/` and adds lines to
`results/crm-prototype-client/runs.csv`; nothing from an earlier run is overwritten.

Run: `./run.sh crm-prototype-client` (from `notebooks/`), or Run All here and Save As into the iteration folder.

In [1]:
APP = 'crm-prototype-client'   # this notebook's project (folder under apps/)
RUN = ''        # iteration folder name; empty = next free iter-NNN (run.sh sets it)

In [2]:
import sys; sys.path.insert(0, '.')
import pandas as pd
import coverage_lab as C
import ref_javascript
REFERENCES = [ref_javascript]   # this project's languages that get a functions-expected count
pd.set_option('display.max_rows', 300); pd.set_option('display.max_colwidth', 120)
RUN = RUN or C.next_run_id(APP)
OUT = C.run_dir(APP, RUN)   # results/<APP>/runs/<RUN>/
stamp = C.run_stamp(APP)
{**stamp, 'run': RUN}

$ git rev-parse --short HEAD 
$ /Users/thiru/.local/bin/code-review-graph --version 


{'app': 'crm-prototype-client',
 'app_commit': '325788b',
 'tool': 'code-review-graph 2.3.8',
 'run_at': '2026-11-17T20:23:28',
 'run': 'iter-001_2026-11-17_2023'}

## 1. Build the graph exactly as Synthetix does
`CRG_DATA_DIR=<dir> code-review-graph build --repo <root> -q`, after removing the repo from the tool's registry.

In [3]:
db = C.build(APP, OUT)
db

$ /Users/thiru/.local/bin/code-review-graph repos 
$ /Users/thiru/.local/bin/code-review-graph build --repo /Users/thiru/MySpace/Celsior/repo/code-graph-rnd/apps/crm-prototype-client -q (env {'CRG_DATA_DIR': '/Users/thiru/MySpace/Celsior/repo/code-graph-rnd/results/crm-prototype-client/runs/iter-001_2026-11-17_2023/crg'})


PosixPath('/Users/thiru/MySpace/Celsior/repo/code-graph-rnd/results/crm-prototype-client/runs/iter-001_2026-11-17_2023/crg/graph.db')

## 2. Sheet 1 — one row per code file
Read? / Why come from the tool's own filter and a re-parse with the same grammar.

In [4]:
rows, funcs, left_out = C.coverage(APP, db, REFERENCES)
sheet1 = pd.DataFrame([{k: v for k, v in r.__dict__.items() if k != 'expected'} for r in rows])
sheet1.read.value_counts()

$ git ls-files 
$ node functions.mjs crm-prototype-client (157 files)


read
Fully read    157
Name: count, dtype: int64

### Files left out of sheet 1 (not code)
No language in the tool and not a programming-language extension: docs, licences, images, Dockerfiles, config.

In [5]:
print(len(left_out), 'files left out')
pd.Series([p.rsplit('.', 1)[-1] if '.' in p.rsplit('/', 1)[-1] else '(no extension)' for p in left_out]).value_counts()

15 files left out


json          7
md            2
gitignore     1
prettierrc    1
html          1
png           1
svg           1
css           1
Name: count, dtype: int64

In [6]:
sheet1[sheet1.read != 'Fully read'][['file', 'language', 'lines', 'read', 'why', 'lines_read', 'notes']]

,file,language,lines,read,why,lines_read,notes


## 3. Functions: expected (language's own parser) vs found (graph.db)

In [7]:
matched = C.match_functions(rows, funcs)
matched.groupby(['language', 'status']).size()

language    status   
javascript  found        545
            not found     78
dtype: int64

In [8]:
matched[matched.status != 'found']

,language,file,function,line,status
36,javascript,src/components/common/RowActionsMenu.jsx,onSuccess,62,not found
37,javascript,src/components/common/RowActionsMenu.jsx,onError,66,not found
79,javascript,src/components/forms/leads/ContactPersonForm.jsx,ContactPersonForm,29,not found
81,javascript,src/components/forms/leads/LeadForm.jsx,setContactRef,60,not found
97,javascript,src/components/forms/pipeline/ProposalUploadSection.jsx,validate,156,not found
99,javascript,src/components/forms/prospectus/ProspectusForm.jsx,setContactRef,55,not found
110,javascript,src/components/pipeline/PipelineCard.jsx,collect,33,not found
114,javascript,src/components/pipeline/PipelineColumn.jsx,drop,20,not found
115,javascript,src/components/pipeline/PipelineColumn.jsx,canDrop,25,not found
116,javascript,src/components/pipeline/PipelineColumn.jsx,collect,26,not found


### Files the reference parser could not fully check
Listed, never silently dropped (a syntax error, or code the reference cannot parse).

In [9]:
C.could_not_check(rows)

,language,file,reason


### Unnamed functions (callbacks, lambdas) — not counted as expected
Shown per file with the lines they cover. Their code is still walked by the tool; only a separate row is missing.

In [10]:
cb = C.callbacks_table(rows)
print(cb.callbacks.sum(), 'unnamed functions,', cb.lines.sum(), 'lines')
cb.sort_values('callbacks', ascending=False)

438 unnamed functions, 5350 lines


,language,file,callbacks,lines
18,javascript,src/components/forms/call-logs/CallLogForm.jsx,34,761
56,javascript,src/pages/LeadsPage/LeadsPage.jsx,32,247
67,javascript,src/pages/ProspectusPage/ProspectusListPage.jsx,30,224
47,javascript,src/pages/ArchivedDataPage/ArchivedDataPage.jsx,23,402
49,javascript,src/pages/CallLogsPage/CallLogsPage.jsx,20,190
22,javascript,src/components/forms/pipeline/OpportunityForm.jsx,17,272
64,javascript,src/pages/ProspectusPage/BulkUploadProspectusPage.jsx,15,93
66,javascript,src/pages/ProspectusPage/ProspectusDetailPage.jsx,15,181
21,javascript,src/components/forms/leads/LeadForm.jsx,15,274
79,javascript,src/utils/prospectusTransformers.js,11,123


## 4. Sheet 2 — per language

In [11]:
sheet2 = C.parsers_table(APP, rows, matched, RUN, REFERENCES)
sheet2

,Language,Package,Repo,Files tried,Files parsed,Functions expected,Functions found,Evidence,Notes
0,javascript,code-review-graph 2.3.8 (grammar: javascript),crm-prototype-client,157,157,623,545,results/crm-prototype-client/runs/iter-001_2026-11-17_2023/coverage.ipynb,expected from TypeScript compiler 5.8.3 (js-ref); lost to same-name overwrite: 0; not found: 78; unnamed functions n...


## 5. Sheet 3 — summary (same formulas as the workbook)

In [12]:
s = sheet1
summary = {'Files': len(s), 'Fully read': (s.read == 'Fully read').sum(), 'Partly read': (s.read == 'Partly read').sum(),
  'Not read': (s.read == 'Not read').sum(), 'Lines': int(s.lines.fillna(0).sum()), 'Lines read': int(s.lines_read.fillna(0).sum()),
  'Not shown to user': ((s.read != 'Fully read') & (s.shown == 'No')).sum()}
summary['% lines read'] = round(100 * summary['Lines read'] / summary['Lines'], 1)
summary

{'Files': 157,
 'Fully read': np.int64(157),
 'Partly read': np.int64(0),
 'Not read': np.int64(0),
 'Lines': 23716,
 'Lines read': 23716,
 'Not shown to user': np.int64(0),
 '% lines read': 100.0}

In [13]:
s[s.read == 'Not read'].why.value_counts()

Series([], Name: count, dtype: int64)

## 6. Write the workbook

In [14]:
C.write_workbook(APP, rows, sheet2, OUT)

/Users/thiru/MySpace/Celsior/repo/code-graph-rnd/.venv/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


PosixPath('/Users/thiru/MySpace/Celsior/repo/code-graph-rnd/results/crm-prototype-client/runs/iter-001_2026-11-17_2023/Ingestion_Coverage_crm-prototype-client.xlsx')

## 7. Add this run to the repo's run log
`results/<repo>/runs.csv`, append-only: one line per run (per language with a reference parser).

In [15]:
log = C.log_run(APP, RUN, stamp, summary, sheet2)
pd.read_csv(log)

,run,run_at,app_commit,tool,files,fully_read,partly_read,not_read,lines,lines_read,pct_lines_read,language,functions_expected,functions_found,notes
0,iter-001_2026-11-17_2023,2026-11-17T20:23:28,325788b,code-review-graph 2.3.8,157,157,0,0,23716,23716,100.0,javascript,623,545,NaN
